In [ ]:
import io, json, threading, os
import numpy as np
import h5py
from scipy.interpolate import griddata
import matplotlib as mpl
from http.server import HTTPServer, BaseHTTPRequestHandler
import socketserver
from IPython.display import HTML

# ── Configuration ──
H5_PATH = '/mnt/c/Users/photo/Photonics_Group/Ruihuan/kubeflow-tdgl/notebooks/sim_output.h5'
NX, NY = 100, 50  # interpolation grid size
PREFETCH_AHEAD = 5
KEEP_BEHIND = 2
PLAYBACK_INTERVAL_MS = 100

print(f'HDF5: {H5_PATH}')
print(f'Exists: {os.path.exists(H5_PATH)}')

In [ ]:
class FrameSource:
    """Reads individual frames from an HDF5 file."""

    def __init__(self, h5_path: str, points: np.ndarray,
                 grid_pts: np.ndarray, nx: int, ny: int,
                 total_frames: int = None):
        self.h5_path = h5_path
        self.points = points
        self.grid_pts = grid_pts
        self.nx = nx
        self.ny = ny
        if total_frames is None:
            with h5py.File(h5_path, 'r') as f:
                self._total_frames = len(f['data'].keys())
        else:
            self._total_frames = total_frames

    def frame_exists(self, idx: int) -> bool:
        try:
            with h5py.File(self.h5_path, 'r') as f:
                return str(idx) in f['data']
        except Exception:
            return False

    def load_frame(self, idx: int) -> np.ndarray:
        """Load frame idx, return interpolated 2D array (ny, nx)."""
        with h5py.File(self.h5_path, 'r') as f:
            psi = np.abs(np.array(f[f'data/{idx}/psi']))
        Z = griddata(self.points, psi, self.grid_pts, method='cubic',
                     fill_value=0.0).reshape(self.ny, self.nx)
        return np.clip(Z, 0, None)

    @property
    def latest_available(self) -> int:
        try:
            with h5py.File(self.h5_path, 'r') as f:
                return len(f['data'].keys())
        except Exception:
            return 0

    @property
    def total_frames(self) -> int:
        return self._total_frames

    def build_interpolation_grid(self):
        """Return (gx, gy) arrays for heatmap axes."""
        xmin, xmax = self.points[:, 0].min(), self.points[:, 0].max()
        ymin, ymax = self.points[:, 1].min(), self.points[:, 1].max()
        return (np.linspace(xmin, xmax, self.nx),
                np.linspace(ymin, ymax, self.ny))

print('FrameSource defined')

In [ ]:
class FrameCache:
    """Sliding window cache with background prefetch."""

    def __init__(self, source: FrameSource,
                 prefetch_ahead: int = 5, keep_behind: int = 2):
        self.source = source
        self.prefetch_ahead = prefetch_ahead
        self.keep_behind = keep_behind
        self._cache: dict[int, np.ndarray] = {}
        self._lock = threading.Lock()

    def get(self, idx: int) -> np.ndarray | None:
        with self._lock:
            return self._cache.get(idx)

    def ensure(self, idx: int) -> np.ndarray:
        """Blocking: load frame if not cached, prefetch ahead, evict behind."""
        with self._lock:
            if idx in self._cache:
                frame = self._cache[idx]
            else:
                frame = None

        if frame is None:
            frame = self.source.load_frame(idx)
            with self._lock:
                self._cache[idx] = frame

        self.evict(idx - self.keep_behind)
        self.prefetch(idx + 1, self.prefetch_ahead)
        return frame

    def prefetch(self, from_idx: int, count: int) -> None:
        """Background thread: load count frames starting from from_idx."""
        def _worker():
            for i in range(from_idx, from_idx + count):
                if i >= self.source.total_frames:
                    break
                with self._lock:
                    if i in self._cache:
                        continue
                if not self.source.frame_exists(i):
                    continue
                frame = self.source.load_frame(i)
                with self._lock:
                    self._cache[i] = frame

        t = threading.Thread(target=_worker, daemon=True)
        t.start()

    def evict(self, before_idx: int) -> None:
        """Remove frames with index < before_idx."""
        with self._lock:
            to_remove = [k for k in self._cache if k < before_idx]
            for k in to_remove:
                del self._cache[k]

    @property
    def cached_indices(self) -> list[int]:
        with self._lock:
            return sorted(self._cache.keys())

print('FrameCache defined')

In [ ]:
PLAYER_HTML = r"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>
*{margin:0;padding:0;box-sizing:border-box}
body{background:#1e1e1e;color:#eee;font-family:system-ui,sans-serif;padding:10px}
.p{max-width:720px;margin:0 auto}
.c{display:flex;align-items:center;gap:8px;margin-bottom:6px}
button{width:40px;height:32px;border:none;border-radius:4px;background:#0f3460;color:#fff;font-size:16px;cursor:pointer}
button:hover{background:#16537e}
input[type=range]{flex:1;height:5px;-webkit-appearance:none;background:#444;border-radius:3px;outline:none}
input[type=range]::-webkit-slider-thumb{-webkit-appearance:none;width:16px;height:16px;border-radius:50%;background:#e94560;cursor:pointer}
.l{font-size:12px;min-width:90px;text-align:right;color:#aaa}
canvas{width:100%;border-radius:4px;display:block}
.r{font-size:11px;color:#555;margin-top:4px;text-align:center}
</style></head><body>
<div class="p">
 <div class="c">
  <button id="b">&#9654;</button>
  <input type="range" id="s" min="0" max="0" value="0">
  <span class="l" id="n">0 / ?</span>
 </div>
 <canvas id="cv"></canvas>
 <div class="r">|&psi;| range: [0.00, 1.05]</div>
</div>
<script>
(function(){
var cv=document.getElementById('cv'),cx=cv.getContext('2d'),
    b=document.getElementById('b'),s=document.getElementById('s'),
    n=document.getElementById('n');
var NX=0,NY=0,TOT=0,playing=false,cur=-1,INT=100,tm=null,ep=0;

fetch('/info').then(function(r){return r.json()}).then(function(d){
  NX=d.nx;NY=d.ny;TOT=d.total;
  cv.width=NX;cv.height=NY;
  s.max=TOT-1;n.textContent='0 / '+TOT;
  loadAndDraw(0);
  setInterval(function(){
    fetch('/info').then(function(r){return r.json()}).then(function(d){
      if(d.available>0) s.max=d.available-1;
    });
  },2000);
}).catch(function(e){console.error(e)});

function loadAndDraw(i){
  if(i<0||i>=TOT) return;
  cur=i;s.value=i;n.textContent=i+' / '+TOT;
  var e=++ep;
  fetch('/frame/'+i).then(function(r){
    if(e!==ep)return null;
    if(!r.ok)return null;
    return r.arrayBuffer();
  }).then(function(buf){
    if(e!==ep||!buf||buf.byteLength!==NX*NY*4)return;
    cx.putImageData(new ImageData(new Uint8ClampedArray(buf),NX,NY),0,0);
  }).catch(function(){});
}

b.onclick=function(){
  playing=!playing;
  b.innerHTML=playing?'&#10074;&#10074;':'&#9654;';
  if(playing)tick();else clearTimeout(tm);
};

function tick(){
  if(!playing)return;
  var nx=cur+1;
  if(nx>=TOT){playing=false;b.innerHTML='&#9654;';return;}
  var e=++ep;
  fetch('/frame/'+nx).then(function(r){
    if(e!==ep)return null;
    if(!r.ok)throw'wait';
    return r.arrayBuffer();
  }).then(function(buf){
    if(e!==ep)return;
    if(!buf||buf.byteLength!==NX*NY*4)throw'bad';
    cx.putImageData(new ImageData(new Uint8ClampedArray(buf),NX,NY),0,0);
    cur=nx;s.value=nx;n.textContent=nx+' / '+TOT;
    tm=setTimeout(tick,INT);
  }).catch(function(){
    if(e!==ep)return;
    tm=setTimeout(tick,200);
  });
}

s.oninput=function(){
  clearTimeout(tm);
  loadAndDraw(parseInt(s.value));
  if(playing)tm=setTimeout(tick,INT);
};
})();
</script></body></html>"""


class _Handler(BaseHTTPRequestHandler):
    source = None
    cache = None
    cmap = None
    vmin = 0.0
    vmax = 1.05

    def do_GET(self):
        if self.path in ('/', '/index.html'):
            self._reply(PLAYER_HTML.encode(), 'text/html')
        elif self.path.startswith('/frame/'):
            try:
                idx = int(self.path.split('/')[-1])
                if not self.source.frame_exists(idx):
                    self.send_response(404); self.end_headers(); return
                Z = self.cache.ensure(idx)
                norm = np.clip((Z - self.vmin) / (self.vmax - self.vmin), 0, 1)
                rgba = (self.cmap(norm) * 255).astype(np.uint8)
                self._reply(rgba.tobytes(), 'application/octet-stream')
            except Exception:
                self.send_response(500); self.end_headers()
        elif self.path == '/info':
            self._reply(json.dumps({
                'total': self.source.total_frames,
                'available': self.source.latest_available,
                'nx': self.source.nx, 'ny': self.source.ny,
            }).encode(), 'application/json')
        else:
            self.send_response(404); self.end_headers()

    def _reply(self, data, ct):
        self.send_response(200)
        self.send_header('Content-Type', ct)
        self.send_header('Content-Length', str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *a): pass


class _TS(socketserver.ThreadingMixIn, HTTPServer):
    daemon_threads = True


class FrameServer:
    def __init__(self, source, cache, port=8765):
        self.source = source
        self.cache = cache
        self._port = port
        self._cmap = mpl.colormaps['inferno']
        self._srv = None

    def start(self):
        _Handler.source = self.source
        _Handler.cache = self.cache
        _Handler.cmap = self._cmap
        self._srv = _TS(('0.0.0.0', self._port), _Handler)
        self._port = self._srv.server_address[1]
        threading.Thread(target=self._srv.serve_forever, daemon=True).start()
        return self._port

    def stop(self):
        if self._srv: self._srv.shutdown()

    @property
    def url(self):
        return f'http://localhost:{self._port}/'

print('FrameServer defined')

In [ ]:
# ── Instantiate and start ──

with h5py.File(H5_PATH, 'r') as f:
    _points = np.array(f['solution/device/mesh/sites'])

_xmin, _xmax = _points[:, 0].min(), _points[:, 0].max()
_ymin, _ymax = _points[:, 1].min(), _points[:, 1].max()
_gx = np.linspace(_xmin, _xmax, NX)
_gy = np.linspace(_ymin, _ymax, NY)
_GX, _GY = np.meshgrid(_gx, _gy)
_grid_pts = np.column_stack([_GX.ravel(), _GY.ravel()])

source = FrameSource(H5_PATH, _points, _grid_pts, NX, NY)
cache = FrameCache(source, prefetch_ahead=PREFETCH_AHEAD, keep_behind=KEEP_BEHIND)
server = FrameServer(source, cache, port=8765)
port = server.start()

print(f'Source: {source.total_frames} frames, {source.latest_available} available')
print(f'Grid: {_points.shape[0]} mesh points -> {NX}x{NY} interpolation')
print(f'Server: {server.url}')
print()
print('>>> Click the link below to open the player in a new browser tab <<<')
HTML(f'<a href="{server.url}" target="_blank" style="font-size:18px;color:#4fc3f7;">▶ Open Frame Player (port {port})</a>')